#model upgradation

In [1]:
pip install pandas scikit-learn xgboost sentence-transformers jellyfish fuzzywuzzy[speedup] tqdm

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import numpy as np
import re
import jellyfish
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from fuzzywuzzy import fuzz
from sklearn.preprocessing import normalize
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from tqdm import tqdm

tqdm.pandas()

def clean_name(name):
    if pd.isnull(name):
        return ""

    # Convert to lowercase
    name = name.lower()

    # Add space after periods if missing (e.g., "dr.sravan" → "dr. sravan")
    name = re.sub(r'\.(?=\w)', '. ', name)

    # Remove common prefixes
    prefixes = ['dr', 'mr', 'mrs', 'ms', 'miss', 'prof', 'sir', 'madam', 'shri', 'smt', 'doctor', 'professor']
    words = name.split()
    while words and words[0].strip('.') in prefixes:
        words.pop(0)
    name = ' '.join(words)

    # Convert initials with dots or glued (e.g., "m.k." or "mk") into spaced form
    name = re.sub(r'\b([a-z])\.', r'\1', name)                # "m.k." → "mk"
    name = re.sub(r'\b([a-z]{2,3})\b', lambda m: ' '.join(m.group(1)) if len(m.group(1)) <= 3 else m.group(1), name)

    # Remove non-alphabet characters (except space)
    name = re.sub(r'[^a-z\s]', '', name)

    # Normalize spaces
    name = re.sub(r'\s+', ' ', name).strip()

    # Remove duplicate consecutive words
    tokens = name.split()
    deduped_tokens = [t for i, t in enumerate(tokens) if i == 0 or t != tokens[i - 1]]
    return ' '.join(deduped_tokens)

def longest_common_substring(s1, s2):
    m = [[0]*(1+len(s2)) for _ in range(1+len(s1))]
    longest = 0
    for i in range(1, 1+len(s1)):
        for j in range(1, 1+len(s2)):
            if s1[i-1] == s2[j-1]:
                m[i][j] = m[i-1][j-1] + 1
                longest = max(longest, m[i][j])
            else:
                m[i][j] = 0
    return longest

def jaccard_similarity(a, b):
    set1, set2 = set(a.split()), set(b.split())
    return len(set1 & set2) / len(set1 | set2) if set1 | set2 else 0.0

def ngram_overlap(a, b, n=3):
    ngrams = lambda s: {s[i:i+n] for i in range(len(s)-n+1)} if len(s) >= n else set()
    ng1, ng2 = ngrams(a), ngrams(b)
    return len(ng1 & ng2) / len(ng1 | ng2) if ng1 | ng2 else 0.0

def compute_features(df, tfidf, sbert):
    df['name1_clean'] = df['name1'].progress_apply(clean_name)
    df['name2_clean'] = df['name2'].progress_apply(clean_name)

    # TF-IDF cosine sim
    tfidf1 = tfidf.transform(df['name1_clean'])
    tfidf2 = tfidf.transform(df['name2_clean'])
    df['cosine_sim'] = [cosine_similarity(tfidf1[i], tfidf2[i])[0][0] for i in range(tfidf1.shape[0])]

    # SBERT
    embeds1 = normalize(sbert.encode(df['name1_clean'].tolist(), show_progress_bar=True))
    embeds2 = normalize(sbert.encode(df['name2_clean'].tolist(), show_progress_bar=True))
    df['sbert_sim'] = np.einsum('ij,ij->i', embeds1, embeds2)

    # Jaro-Winkler
    df['jaro_sim'] = df.progress_apply(lambda row: jellyfish.jaro_winkler_similarity(row['name1_clean'], row['name2_clean']), axis=1)

    # Levenshtein
    df['levenshtein'] = df.progress_apply(lambda row: jellyfish.levenshtein_distance(row['name1_clean'], row['name2_clean']), axis=1)

    # First letter match
    df['first_letter_match'] = (df['name1_clean'].str[0] == df['name2_clean'].str[0]).astype(int)

    # Length diff
    df['len_diff'] = (df['name1_clean'].str.len() - df['name2_clean'].str.len()).abs()

    # Soundex
    df['soundex_match'] = df.progress_apply(lambda row: int(jellyfish.soundex(row['name1_clean']) == jellyfish.soundex(row['name2_clean'])), axis=1)

    # Metaphone
    df['metaphone_match'] = df.progress_apply(lambda row: int(jellyfish.metaphone(row['name1_clean']) == jellyfish.metaphone(row['name2_clean'])), axis=1)

    # Token Set Ratio
    df['token_set_ratio'] = df.progress_apply(lambda row: fuzz.token_set_ratio(row['name1_clean'], row['name2_clean']) / 100.0, axis=1)

    # Additional features
    df['lcs'] = df.progress_apply(lambda row: longest_common_substring(row['name1_clean'], row['name2_clean']), axis=1)
    df['jaccard'] = df.progress_apply(lambda row: jaccard_similarity(row['name1_clean'], row['name2_clean']), axis=1)
    df['ngram_overlap'] = df.progress_apply(lambda row: ngram_overlap(row['name1_clean'], row['name2_clean']), axis=1)

    return df

# Load and prepare
df = pd.read_csv("/content/dataset_for_training.csv")  # Must include name1, name2, is_match
df['is_match'] = df['is_match'].astype(int)

# Fit vectorizer and SBERT
tfidf = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4))
tfidf.fit(pd.concat([df['name1'].dropna(), df['name2'].dropna()]).apply(clean_name))

sbert = SentenceTransformer('all-MiniLM-L6-v2')

# Feature extraction
df = compute_features(df, tfidf, sbert)

# Feature list
features = [
    'cosine_sim', 'sbert_sim', 'jaro_sim', 'levenshtein',
    'first_letter_match', 'len_diff', 'soundex_match', 'lcs',
    'metaphone_match', 'token_set_ratio', 'jaccard', 'ngram_overlap'
]

X = df[features]
y = df['is_match']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = XGBClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, use_label_encoder=False, eval_metric='logloss')
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_val)
print("✅ Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))

# Save everything
joblib.dump(model, "xgb_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
joblib.dump(sbert, "sbert_model.pkl")
joblib.dump(features, "features_used.pkl")
print("✅ Model and feature list saved!")


ImportError: cannot import name 'Tensor' from 'torch' (unknown location)

In [2]:
import pandas as pd
import numpy as np
import joblib
import jellyfish
import re
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from fuzzywuzzy import fuzz
from tqdm import tqdm
import matplotlib.pyplot as plt
import xgboost as xgb
import shap

tqdm.pandas()

# ----------- Utility Functions -----------

def clean_name(name):
    if pd.isnull(name):
        return ""
    name = name.lower()
    name = re.sub(r'\.(?=\w)', '. ', name)  # Add space after dot if missing
    prefixes = {'dr', 'mr', 'mrs', 'ms', 'miss', 'prof', 'sir', 'madam', 'shri', 'smt', 'doctor', 'professor'}
    words = name.split()
    while words and words[0].strip('.') in prefixes:
        words.pop(0)
    name = ' '.join(words)
    name = re.sub(r'[^a-z\s]', '', name)  # Remove non-alpha chars except space
    name = re.sub(r'\s+', ' ', name).strip()
    tokens = name.split()
    deduped = [tokens[0]] if tokens else []
    for token in tokens[1:]:
        if token != deduped[-1]:
            deduped.append(token)
    return ' '.join(deduped)

def longest_common_substring(a, b):
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    result = 0
    for i in range(m):
        for j in range(n):
            if a[i] == b[j]:
                dp[i+1][j+1] = dp[i][j] + 1
                result = max(result, dp[i+1][j+1])
    return result

def jaccard_similarity(a, b):
    set1, set2 = set(a.split()), set(b.split())
    if not set1 or not set2:
        return 0.0
    return len(set1 & set2) / len(set1 | set2)

def ngram_overlap(a, b, n=3):
    def ngrams(s):
        return {s[i:i+n] for i in range(len(s) - n + 1)} if len(s) >= n else set()
    ng1, ng2 = ngrams(a), ngrams(b)
    if not ng1 or not ng2:
        return 0.0
    return len(ng1 & ng2) / len(ng1 | ng2)

# ----------- Feature Computation Function -----------

def compute_features(df, tfidf, sbert):
    print("🧹 Cleaning names...")
    name1_clean = df['name1'].progress_apply(clean_name)
    name2_clean = df['name2'].progress_apply(clean_name)

    print("📊 Computing TF-IDF cosine similarity...")
    tfidf1 = tfidf.transform(name1_clean)
    tfidf2 = tfidf.transform(name2_clean)
    cosine_sim = [cosine_similarity(tfidf1[i], tfidf2[i])[0][0] for i in range(tfidf1.shape[0])]

    print("🧠 Computing SBERT cosine similarity...")
    embeds1 = normalize(sbert.encode(name1_clean.tolist(), show_progress_bar=True))
    embeds2 = normalize(sbert.encode(name2_clean.tolist(), show_progress_bar=True))
    sbert_sim = np.einsum('ij,ij->i', embeds1, embeds2)

    print("🔢 Calculating other string similarity features...")
    jaro_sim = df.progress_apply(lambda row: jellyfish.jaro_winkler_similarity(clean_name(row['name1']), clean_name(row['name2'])), axis=1)
    levenshtein = df.progress_apply(lambda row: jellyfish.levenshtein_distance(clean_name(row['name1']), clean_name(row['name2'])), axis=1)
    first_letter_match = (name1_clean.str[0] == name2_clean.str[0]).astype(int)
    len_diff = (name1_clean.str.len() - name2_clean.str.len()).abs()
    soundex_match = df.progress_apply(lambda row: int(jellyfish.soundex(clean_name(row['name1'])) == jellyfish.soundex(clean_name(row['name2']))), axis=1)
    metaphone_match = df.progress_apply(lambda row: int(jellyfish.metaphone(clean_name(row['name1'])) == jellyfish.metaphone(clean_name(row['name2']))), axis=1)
    token_set_ratio = df.progress_apply(lambda row: fuzz.token_set_ratio(clean_name(row['name1']), clean_name(row['name2'])) / 100.0, axis=1)
    lcs = df.progress_apply(lambda row: longest_common_substring(clean_name(row['name1']), clean_name(row['name2'])), axis=1)
    jaccard = df.progress_apply(lambda row: jaccard_similarity(clean_name(row['name1']), clean_name(row['name2'])), axis=1)
    ngram = df.progress_apply(lambda row: ngram_overlap(clean_name(row['name1']), clean_name(row['name2'])), axis=1)

    features = pd.DataFrame({
        'cosine_sim': cosine_sim,
        'sbert_sim': sbert_sim,
        'jaro_sim': jaro_sim,
        'levenshtein': levenshtein,
        'first_letter_match': first_letter_match,
        'len_diff': len_diff,
        'soundex_match': soundex_match,
        'metaphone_match': metaphone_match,
        'token_set_ratio': token_set_ratio,
        'lcs': lcs,
        'jaccard': jaccard,
        'ngram_overlap': ngram
    })

    df = pd.concat([df, features], axis=1)
    return df

# ----------- Main Execution -----------

def main():
    print("🚀 Loading models and vectorizers...")
    model = joblib.load("xgb_model.pkl")
    tfidf = joblib.load("tfidf_vectorizer.pkl")
    sbert = joblib.load("sbert_model.pkl")
    feature_list = joblib.load("features_used.pkl")

    print("📥 Loading test dataset...")
    df_test = pd.read_csv('/content/MCI.sanofi2lakhs_nm_st.csv')

    print("⚙️ Computing features for test data...")
    df_test = compute_features(df_test, tfidf, sbert)

    # Ensure all model features exist in test data, add 0 if missing
    for feat in feature_list:
        if feat not in df_test.columns:
            df_test[feat] = 0

    X_test = df_test[feature_list]

    print("🤖 Making predictions...")
    df_test['predicted_match'] = model.predict(X_test)

    if 'is_match' in df_test.columns:
        from sklearn.metrics import accuracy_score, classification_report
        print("\n📈 Evaluation on test set:")
        print(f"Accuracy: {accuracy_score(df_test['is_match'], df_test['predicted_match']):.4f}")
        print(classification_report(df_test['is_match'], df_test['predicted_match']))

    print("💾 Saving predictions...")
    df_test.to_csv("new_data_output.csv", index=False)
    print("✅ Predictions saved to new_data_output.csv")

    # -------- Feature Importance Plots --------
    print("\n📊 Plotting XGBoost feature importance...")
    xgb.plot_importance(model, importance_type='gain', max_num_features=20, height=0.5)
    plt.title("Top 20 Feature Importances (XGBoost)")
    plt.tight_layout()
    plt.show()

    print("\n🔍 Running SHAP analysis for interpretability...")
    explainer = shap.Explainer(model)
    shap_values = explainer(X_test)

    print("📊 Displaying SHAP summary plot...")
    shap.summary_plot(shap_values, X_test, max_display=20)

if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'shap'

In [3]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

# True labels and predicted outputs
y_true = df_test['is_match']
y_pred = df_test['predicted_match']

# Predicted probabilities for positive class
if hasattr(model, "predict_proba"):
    y_proba = model.predict_proba(X_test)[:, 1]
else:
    # If no predict_proba, fallback to predictions (not ideal for ROC/PR curves)
    y_proba = y_pred

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
fig, axs = plt.subplots(1, 3, figsize=(18, 5))

# Plot Confusion Matrix
disp.plot(ax=axs[0], cmap=plt.cm.Blues)
axs[0].set_title('Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_proba)
roc_auc = auc(fpr, tpr)
axs[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
axs[1].plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
axs[1].set_xlim([0.0, 1.0])
axs[1].set_ylim([0.0, 1.05])
axs[1].set_xlabel('False Positive Rate')
axs[1].set_ylabel('True Positive Rate')
axs[1].set_title('Receiver Operating Characteristic (ROC)')
axs[1].legend(loc="lower right")

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_true, y_proba)
ap_score = average_precision_score(y_true, y_proba)
axs[2].step(recall, precision, color='b', alpha=0.8, where='post')
axs[2].set_xlabel('Recall')
axs[2].set_ylabel('Precision')
axs[2].set_ylim([0.0, 1.05])
axs[2].set_xlim([0.0, 1.0])
axs[2].set_title(f'Precision-Recall Curve (AP={ap_score:.2f})')

plt.tight_layout()
plt.show()


NameError: name 'df_test' is not defined